# 04 - Build the Star Schema

In this notebook, we will turn the cleaned customer churn dataset into separate dimension and fact tables.

This is the first major modelling step of the project.

We will create:

- `dim_customer`
- `dim_contract`
- `dim_service`
- `dim_payment_method`
- `dim_churn_reason`
- `fact_customer_snapshot`

The output files will be saved in `data/processed/star_schema`.

## 1. Import Libraries and Load Data

We start from the cleaned CSV created in notebook 02.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
clean_file = Path("../data/processed/telco_customer_churn_clean.csv")
schema_dir = Path("../data/processed/star_schema")

schema_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(clean_file)

df.head()

,customerid,count,country,state,city,zip_code,lat_long,latitude,longitude,gender,...,monthly_charges,total_charges,churn_label,churn_value,churn_score,cltv,churn_reason,is_churned,is_month_to_month,revenue_at_risk
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,53.85,108.15,Yes,1,86,3239,Competitor made better offer,True,True,53.85
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,70.70,151.65,Yes,1,67,2701,Moved,True,True,70.70
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,99.65,820.50,Yes,1,86,5372,Moved,True,True,99.65
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,104.80,3046.05,Yes,1,84,5003,Moved,True,True,104.80
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,103.70,5036.30,Yes,1,89,5340,Competitor had better devices,True,True,103.70


## 2. Fix Missing `total_charges`

In the cleaning notebook, we found 11 missing values in `total_charges`.

Before filling them, inspect those rows. A common reason is that these customers have `tenure_months = 0`, meaning they are brand-new customers with no accumulated charges yet.

In [3]:
df[df["total_charges"].isna()][[
    "customerid",
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "churn_label",
]]

,customerid,tenure_months,monthly_charges,total_charges,churn_label
2234,4472-LVYGI,0,52.55,NaN,No
2438,3115-CZMZD,0,20.25,NaN,No
2568,5709-LVOEQ,0,80.85,NaN,No
2667,4367-NUYAO,0,25.75,NaN,No
2856,1371-DWPAZ,0,56.05,NaN,No
4331,7644-OMVMY,0,19.85,NaN,No
4687,3213-VVOLG,0,25.35,NaN,No
5104,2520-SGTTA,0,20.00,NaN,No
5719,2923-ARZLG,0,19.70,NaN,No
6772,4075-WKNIU,0,73.35,NaN,No


**Your notes:**

- Do the missing `total_charges` rows all have `tenure_months = 0`?
- If yes, does filling `total_charges` with `0` make business sense?
- If no, what else might be happening?

In [4]:
df["total_charges"] = df["total_charges"].fillna(0)

df["total_charges"].isna().sum()

0

## 3. Build `dim_customer`

This table describes who and where the customer is.

It keeps one row per customer.

In [5]:
dim_customer = df[[
    "customerid",
    "country",
    "state",
    "city",
    "zip_code",
    "lat_long",
    "latitude",
    "longitude",
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
]].drop_duplicates().reset_index(drop=True)

dim_customer.head()

,customerid,country,state,city,zip_code,lat_long,latitude,longitude,gender,senior_citizen,partner,dependents
0,3668-QPYBK,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No
1,9237-HQITU,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes
2,9305-CDSKC,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes
3,7892-POOKP,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes
4,0280-XJGEX,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes


In [6]:
dim_customer.shape

(7043, 12)

Check: `dim_customer` should have the same number of rows as the original dataset if every customer appears once.

## 4. Build `dim_contract`

This table stores unique contract and billing combinations.

We create a numeric `contract_key` so the fact table can connect to it.

In [7]:
dim_contract = (
    df[["contract", "paperless_billing"]]
    .drop_duplicates()
    .sort_values(["contract", "paperless_billing"])
    .reset_index(drop=True)
)

dim_contract.insert(0, "contract_key", range(1, len(dim_contract) + 1))

dim_contract

,contract_key,contract,paperless_billing
0,1,Month-to-month,No
1,2,Month-to-month,Yes
2,3,One year,No
3,4,One year,Yes
4,5,Two year,No
5,6,Two year,Yes


## 5. Build `dim_payment_method`

This table stores each payment method once.

We also add a simple `payment_risk_classification` field. This is a business assumption, so we should document it later.

In [8]:
dim_payment_method = (
    df[["payment_method"]]
    .drop_duplicates()
    .sort_values("payment_method")
    .reset_index(drop=True)
)

dim_payment_method.insert(0, "payment_method_key", range(1, len(dim_payment_method) + 1))

dim_payment_method["payment_risk_classification"] = dim_payment_method["payment_method"].map({
    "Electronic check": "Higher observed churn risk",
    "Mailed check": "Manual payment method",
    "Bank transfer (automatic)": "Automatic payment method",
    "Credit card (automatic)": "Automatic payment method",
})

dim_payment_method

,payment_method_key,payment_method,payment_risk_classification
0,1,Bank transfer (automatic),Automatic payment method
1,2,Credit card (automatic),Automatic payment method
2,3,Electronic check,Higher observed churn risk
3,4,Mailed check,Manual payment method


**Your notes:**

- Why might electronic check be labelled higher risk?
- Why should we describe this as an assumption instead of a proven fact?

## 6. Build `dim_service`

This table stores unique service combinations.

Because there are many service fields, there are many possible combinations.

In [9]:
service_columns = [
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
]

dim_service = (
    df[service_columns]
    .drop_duplicates()
    .sort_values(service_columns)
    .reset_index(drop=True)
)

dim_service.insert(0, "service_key", range(1, len(dim_service) + 1))

dim_service.head()

,service_key,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies
0,1,No,No phone service,DSL,No,No,No,No,No,No
1,2,No,No phone service,DSL,No,No,No,No,No,Yes
2,3,No,No phone service,DSL,No,No,No,No,Yes,No
3,4,No,No phone service,DSL,No,No,No,No,Yes,Yes
4,5,No,No phone service,DSL,No,No,No,Yes,No,No


In [10]:
dim_service.shape

(322, 10)

## 7. Build `dim_churn_reason`

`churn_reason` is descriptive text, so we will move it into its own dimension.

Customers who did not churn do not have a churn reason. We will label those as `Not churned` so the relationship stays simple.

In [11]:
df["churn_reason_clean"] = df["churn_reason"].fillna("Not churned")

dim_churn_reason = (
    df[["churn_reason_clean"]]
    .drop_duplicates()
    .rename(columns={"churn_reason_clean": "churn_reason"})
    .sort_values("churn_reason")
    .reset_index(drop=True)
)

dim_churn_reason.insert(0, "churn_reason_key", range(1, len(dim_churn_reason) + 1))

dim_churn_reason.head(20)

,churn_reason_key,churn_reason
0,1,Attitude of service provider
1,2,Attitude of support person
2,3,Competitor had better devices
3,4,Competitor made better offer
4,5,Competitor offered higher download speeds
5,6,Competitor offered more data
6,7,Deceased
7,8,Don't know
8,9,Extra data charges
9,10,Lack of affordable download/upload speed


## 8. Add Keys Back to the Main Dataset

Now we merge the generated keys back into the customer-level dataset.

This is how the fact table learns which contract, service, payment method, and churn reason each customer belongs to.

In [12]:
df_with_keys = (
    df
    .merge(dim_contract, on=["contract", "paperless_billing"], how="left")
    .merge(dim_payment_method[["payment_method_key", "payment_method"]], on="payment_method", how="left")
    .merge(dim_service, on=service_columns, how="left")
    .merge(dim_churn_reason, left_on="churn_reason_clean", right_on="churn_reason", how="left", suffixes=("", "_dim"))
)

df_with_keys[[
    "customerid",
    "contract_key",
    "payment_method_key",
    "service_key",
    "churn_reason_key",
]].head()

,customerid,contract_key,payment_method_key,service_key,churn_reason_key
0,3668-QPYBK,2,4,113,4
1,9237-HQITU,2,3,129,14
2,9305-CDSKC,2,3,269,14
3,7892-POOKP,2,3,273,14
4,0280-XJGEX,2,1,285,3


## 9. Build `fact_customer_snapshot`

This fact table keeps one row per customer.

It stores keys plus numeric measures and flags.

In [13]:
fact_customer_snapshot = df_with_keys[[
    "customerid",
    "contract_key",
    "payment_method_key",
    "service_key",
    "churn_reason_key",
    "count",
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "churn_value",
    "churn_score",
    "cltv",
    "is_churned",
    "is_month_to_month",
    "revenue_at_risk",
]].copy()

fact_customer_snapshot.head()

,customerid,contract_key,payment_method_key,service_key,churn_reason_key,count,tenure_months,monthly_charges,total_charges,churn_value,churn_score,cltv,is_churned,is_month_to_month,revenue_at_risk
0,3668-QPYBK,2,4,113,4,1,2,53.85,108.15,1,86,3239,True,True,53.85
1,9237-HQITU,2,3,129,14,1,2,70.70,151.65,1,67,2701,True,True,70.70
2,9305-CDSKC,2,3,269,14,1,8,99.65,820.50,1,86,5372,True,True,99.65
3,7892-POOKP,2,3,273,14,1,28,104.80,3046.05,1,84,5003,True,True,104.80
4,0280-XJGEX,2,1,285,3,1,49,103.70,5036.30,1,89,5340,True,True,103.70


## 10. Validate Table Shapes

Before exporting, check the row counts.

The fact table should have the same number of rows as the original customer dataset.

In [14]:
table_shapes = pd.DataFrame({
    "table": [
        "source_clean_dataset",
        "dim_customer",
        "dim_contract",
        "dim_payment_method",
        "dim_service",
        "dim_churn_reason",
        "fact_customer_snapshot",
    ],
    "rows": [
        len(df),
        len(dim_customer),
        len(dim_contract),
        len(dim_payment_method),
        len(dim_service),
        len(dim_churn_reason),
        len(fact_customer_snapshot),
    ],
})

table_shapes

,table,rows
0,source_clean_dataset,7043
1,dim_customer,7043
2,dim_contract,6
3,dim_payment_method,4
4,dim_service,322
5,dim_churn_reason,21
6,fact_customer_snapshot,7043


**Your notes:**

- Does the fact table row count match the source dataset?
- Why are the contract and payment dimensions much smaller?
- Why is `dim_service` larger than the other small dimensions?

## 11. Export Star Schema Tables

Now we save each table as a CSV file.

These files can be loaded into Power BI or queried with SQL.

In [15]:
dim_customer.to_csv(schema_dir / "dim_customer.csv", index=False)
dim_contract.to_csv(schema_dir / "dim_contract.csv", index=False)
dim_payment_method.to_csv(schema_dir / "dim_payment_method.csv", index=False)
dim_service.to_csv(schema_dir / "dim_service.csv", index=False)
dim_churn_reason.to_csv(schema_dir / "dim_churn_reason.csv", index=False)
fact_customer_snapshot.to_csv(schema_dir / "fact_customer_snapshot.csv", index=False)

sorted(path.name for path in schema_dir.glob("*.csv"))

['dim_churn_reason.csv',
 'dim_contract.csv',
 'dim_customer.csv',
 'dim_payment_method.csv',
 'dim_service.csv',
 'fact_customer_snapshot.csv']

## 12. Next Step

Next, we will use SQL to query these tables.

That will help us validate the model and calculate dashboard metrics before moving into Power BI.

In [16]:
import duckdb

duckdb.__version__

ModuleNotFoundError: No module named 'duckdb'